# Experiment 2 — Resolution Degradation Curves

**Objective:** Run the model selected in Experiment 1 at six resolutions
(640 → 320 in 64-pixel steps) on all three MOT17 sequences.
Compute two primary degradation signals relative to the 640 baseline and plot per-sequence curves.

**Sequences:** MOT17-09 (sparse, pedestrian street, 10.1 ped/fr, low angle),
MOT17-02 (moderate, open square, 31.0 ped/fr, moderate elevation), and
MOT17-04 (dense, pedestrian street at night, 45.3 ped/fr, elevated viewpoint).
Sequences differ in both pedestrian density and camera elevation; these cannot be fully
decoupled within the available MOT17 static-camera sequences.

**Two primary degradation signals (methodology v3):**
1. Identity confusion — IDSW normalised by GT track count (`idsw_per_gt_track`), with ±1-switch noise floor shading
2. End-to-end track continuity — Mostly Tracked ratio (`mostly_tracked_ratio`)

Both signals must be co-plotted in every panel. In dense scenes at low resolution, detection
recall collapse suppresses the association pool and drives raw IDSW counts toward zero even as
`mt_delta_norm` indicates most tracks are lost. Presenting only IDSW would misrepresent the
failure mode as improvement.

Track fragmentation (`frag_ratio`), detection stability, and spatial precision are computed
separately as diagnostic data. They are not primary claims — see methodology v3 for rationale.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import os
import torch
from benchmark.config import (
    DATA_ROOT, RESULTS_RAW, SEQUENCES, SEQ_SUFFIX,
    IMGSZ_BASE,
)
from benchmark.device_profile import load_profile

PROFILE = load_profile(os.environ.get("DEVICE_PROFILE"))
DEVICE  = PROFILE.torch_device

SKIP_EXISTING = True  # Skip evaluation if output CSV already exists (per model, per resolution)

# CLAHE luminance normalisation: applied uniformly across ALL sequences and ALL
# resolutions when True. Parameters are fixed in runner.py (clip=2.0, grid=8×8).
# Produces _clahe-suffixed CSVs; baseline results are untouched.
USE_CLAHE = False

clahe_tag = "_clahe" if USE_CLAHE else ""

print(f"Device      : {PROFILE.device_label}")
print(f"Backend     : {PROFILE.backend}  |  torch device: {DEVICE}")
print(f"Models      : {PROFILE.model_variants}")
print(f"Sequences   : {SEQUENCES}")
print(f"Resolutions : {PROFILE.resolutions}")
print(f"Tag         : {PROFILE.result_tag}")
print(f"CLAHE       : {USE_CLAHE}")

In [ ]:
# Inference loop: 3 models × 6 resolutions × 3 sequences
# CSVs written by notebook 01 are reused when present (skip guard).
from ultralytics import YOLO
from benchmark.runner import run_sequence

for model_path in PROFILE.model_variants:
    print(f"\n── Model: {model_path} ──")
    for imgsz in PROFILE.resolutions:
        for seq_name in SEQUENCES:
            out_csv = RESULTS_RAW / PROFILE.result_tag / "exp1" / f"{seq_name}_{model_path}_{imgsz}{clahe_tag}_{PROFILE.result_tag}.csv"
            if SKIP_EXISTING and out_csv.exists():
                print(f"  {seq_name} @ {imgsz}px: skip")
                continue

            seq_dir = DATA_ROOT / f"{seq_name}-{SEQ_SUFFIX}"
            # Fresh model instance per (model, resolution, sequence) to reset ByteTrack state.
            # .to(DEVICE) places the model on the target device before inference.
            model = YOLO(model_path).to(DEVICE)
            print(f"  {seq_name} @ {imgsz}px: running ...", end=" ", flush=True)
            df = run_sequence(model, seq_dir, imgsz=imgsz, out_csv=out_csv, clahe=USE_CLAHE)
            print(f"done — mean {df['n_detections'].mean():.1f} det/frame")

In [ ]:
# ── Primary signal computation ─────────────────────────────────────────────────
#
# For each (model, sequence, resolution): load the CSV and compute the two primary
# track-continuity signals via track_continuity(). Fragmentation diagnostic
# columns are stored alongside the primary signals but are not plotted.
import pandas as pd
from benchmark.degradation import track_continuity

deg_rows = []

for model_path in PROFILE.model_variants:
    for seq_name in SEQUENCES:
        baseline_csv = RESULTS_RAW / PROFILE.result_tag / "exp1" / f"{seq_name}_{model_path}_{IMGSZ_BASE}{clahe_tag}_{PROFILE.result_tag}.csv"
        seq_dir      = DATA_ROOT / f"{seq_name}-{SEQ_SUFFIX}"

        if not baseline_csv.exists():
            print(f"WARNING: baseline CSV missing for {model_path} / {seq_name}, skip")
            continue

        for imgsz in PROFILE.resolutions:
            csv_path = RESULTS_RAW / PROFILE.result_tag / "exp1" / f"{seq_name}_{model_path}_{imgsz}{clahe_tag}_{PROFILE.result_tag}.csv"
            if not csv_path.exists():
                continue

            tc = track_continuity(csv_path, seq_dir)
            deg_rows.append({
                "model":                model_path,
                "seq":                  seq_name,
                "imgsz":                imgsz,
                "num_switches":         tc["num_switches"],
                "idsw_per_gt_track":    tc["idsw_per_gt_track"],
                "mostly_tracked_ratio": tc["mostly_tracked_ratio"],
                # Fragmentation — diagnostic only, not a primary signal
                "frag_ratio":           tc["frag_ratio"],
                "short_tracks_abs":     tc["short_tracks_abs"],
                "total_initiated":      tc["total_initiated"],
            })

deg_df = pd.DataFrame(deg_rows)

print("Primary signals — absolute values:")
print(deg_df[["model", "seq", "imgsz", "num_switches", "idsw_per_gt_track",
              "mostly_tracked_ratio"]].to_string(index=False))

In [ ]:
# ── Diagnostic signals ─────────────────────────────────────────────────────────
#
# Detection stability, spatial precision, and fragmentation denominator data.
# These are not primary claims and are not plotted in the paper figure.
#
# - det_stability: mean absolute deviation of per-frame detection count vs the
#   640 baseline. Noisy proxy for recall (conflates TP and FP count changes).
# - spatial_prec: mean footpoint displacement vs 640 baseline (pixels).
#   No stable trend distinct from the primary signals was observed.
# - Fragmentation denominator: short_tracks_abs (0–14 events per condition) is
#   too sparse to trend. Retained here to confirm denominator stability.
from benchmark.degradation import detection_stability, spatial_precision

for model_path in PROFILE.model_variants:
    for seq_name in SEQUENCES:
        baseline_csv = RESULTS_RAW / PROFILE.result_tag / "exp1" / f"{seq_name}_{model_path}_{IMGSZ_BASE}{clahe_tag}_{PROFILE.result_tag}.csv"
        if not baseline_csv.exists():
            continue
        for imgsz in PROFILE.resolutions:
            csv_path = RESULTS_RAW / PROFILE.result_tag / "exp1" / f"{seq_name}_{model_path}_{imgsz}{clahe_tag}_{PROFILE.result_tag}.csv"
            if not csv_path.exists():
                continue
            mask = (
                (deg_df["model"] == model_path) &
                (deg_df["seq"]   == seq_name) &
                (deg_df["imgsz"] == imgsz)
            )
            deg_df.loc[mask, "det_stability"] = detection_stability(csv_path, baseline_csv)
            deg_df.loc[mask, "spatial_prec"]  = spatial_precision(csv_path, baseline_csv)

print("Fragmentation denominator diagnostics:")
print(deg_df[["model", "seq", "imgsz", "frag_ratio", "short_tracks_abs",
              "total_initiated"]].to_string(index=False))

In [ ]:
# ── Per-(model, sequence) normalisation anchored at 640 baseline ───────────────
#
# Two primary signals:
# 1. idsw_per_gt_track — identity confusion (positive = more switches = worse)
# 2. mostly_tracked_ratio — end-to-end continuity (inverted so positive = worse)
#
# Noise floor for IDSW: a change of ±1 raw switch, expressed as a fraction of the
# 640 baseline idsw_per_gt_track, defines the minimum detectable effect size given
# the small GT track counts (MOT17-09: 26 tracks, MOT17-02: 53, MOT17-04: 79).
# Changes within this band should not be interpreted as meaningful trends.
import numpy as np
from benchmark.mot_gt import load_gt

n_gt_by_seq = {
    seq: load_gt(DATA_ROOT / f"{seq}-{SEQ_SUFFIX}")["track_id"].nunique()
    for seq in SEQUENCES
}

for model_path in PROFILE.model_variants:
    for seq_name in SEQUENCES:
        mask     = (deg_df["model"] == model_path) & (deg_df["seq"] == seq_name)
        baseline = deg_df[mask & (deg_df["imgsz"] == IMGSZ_BASE)]
        if baseline.empty:
            continue

        base_idsw = float(baseline["idsw_per_gt_track"].iloc[0])
        base_mt   = float(baseline["mostly_tracked_ratio"].iloc[0])
        n_gt      = n_gt_by_seq[seq_name]

        # IDSW relative change from 640 baseline (0 = baseline, positive = more confusion)
        deg_df.loc[mask, "idsw_delta_norm"] = (
            (deg_df.loc[mask, "idsw_per_gt_track"] - base_idsw) / base_idsw
        ).clip(-1, None) if base_idsw > 0 else 0.0

        # ±1-switch noise floor expressed as fraction of baseline idsw_per_gt_track.
        # A ±1 raw switch = ±(1/n_gt) in idsw_per_gt_track units.
        # Normalised to baseline: noise_band = (1/n_gt) / base_idsw
        deg_df.loc[mask, "idsw_noise_band"] = (1.0 / n_gt) / base_idsw if base_idsw > 0 else 0.0

        # MT relative change: inverted so positive = degradation (fewer mostly-tracked)
        deg_df.loc[mask, "mt_delta_norm"] = (
            (base_mt - deg_df.loc[mask, "mostly_tracked_ratio"]) / base_mt
        ).clip(-1, None) if base_mt > 0 else 0.0

print("Primary signals (640 baseline = 0, positive = worse):")
print(deg_df[["model", "seq", "imgsz", "idsw_per_gt_track",
              "idsw_delta_norm", "idsw_noise_band", "mt_delta_norm"]].to_string(index=False))

In [ ]:
# ── Save degradation table as CSV ─────────────────────────────────────────────
# Multi-index (model, imgsz, seq) matching the notebook 01 convention.

deg_out = deg_df.set_index(["model", "imgsz", "seq"])

deg_csv_path = RESULTS_RAW.parent / "degradation" / f"table1_degradation_{PROFILE.result_tag}.csv"
deg_csv_path.parent.mkdir(parents=True, exist_ok=True)
deg_out.to_csv(deg_csv_path)
print(f"Saved to {deg_csv_path}")

In [ ]:
# ── Two-signal degradation plot: IDSW (confusion) + MT (continuity) ───────────
#
# Layout: rows = sequences (sparse / moderate / dense), columns = model variants.
# y-axis is shared across ALL panels (sharey="all") — tick values are identical
# in every panel, so cross-row and cross-column slope comparisons are valid.
#
# Both signals are relative change from the 640-px baseline (0 = baseline).
# Positive = degradation in both cases:
#   IDSW (red solid ■)  — (idsw_current − idsw_640) / idsw_640
#                          positive = more identity switches = worse confusion
#   MT loss (green ▲:) — (mt_640 − mt_current) / mt_640
#                          positive = fewer mostly-tracked GT tracks = worse continuity
#
# NOTE on MT convention: MT ratio decreases as resolution drops (fewer tracks
# covered ≥80%). Plotting (baseline − current)/baseline flips the sign so that
# the green curve rises with degradation — consistent with the IDSW convention.
# A rising green curve means MORE tracks are being lost, not gained.
#
# NOTE on negative IDSW in dense/moderate scenes: at low resolution, detection
# recall collapses. The smaller association pool means fewer opportunities for
# identity switches, so raw IDSW counts fall and the normalized signal goes
# negative. This is NOT an improvement — it is a suppression artefact. MT loss
# (rising green curve) confirms the failure mode in these panels.

import matplotlib.pyplot as plt

SEQ_LABELS = {
    "MOT17-09": "Sparse\n(10.1 ped/fr, low angle)",
    "MOT17-02": "Moderate\n(31.0 ped/fr, moderate elevation)",
    "MOT17-04": "Dense\n(45.3 ped/fr, elevated viewpoint)",
}
MODEL_LABELS = {
    "yolo26n.pt": "YOLOv26-N (nano)",
    "yolo26s.pt": "YOLOv26-S (small)",
    "yolo26m.pt": "YOLOv26-M (medium)",
    "yolo26l.pt": "YOLOv26-L (large)",
    "yolo26x.pt": "YOLOv26-X (xlarge)",
}
COLOR_IDSW = "#d62728"
COLOR_MT   = "#2ca02c"

# Detection-suppression annotation: sequences where population density is high
# enough that detection collapse causes a meaningful IDSW signal reduction.
# MOT17-09 (sparse) stays near zero throughout — small negative excursions there
# are noise, not collapse, so it is excluded from annotation candidates.
SUPPRESS_ANNOT_SEQS   = {"MOT17-02", "MOT17-04"}
SUPPRESS_ANNOT_THRESH = -0.10   # only annotate dips below this value

n_rows = len(SEQUENCES)
n_cols = len(PROFILE.model_variants)

# sharey="all" enforces a single shared y-range across every panel — cross-row
# slope comparisons are visually valid. ylim is set once below.
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(3.5 * n_cols, 3.0 * n_rows),
    sharey="all",
    constrained_layout=True,
)

for row_idx, seq_name in enumerate(SEQUENCES):
    for col_idx, model_path in enumerate(PROFILE.model_variants):
        ax   = axes[row_idx, col_idx]
        mask = (deg_df["model"] == model_path) & (deg_df["seq"] == seq_name)
        sub  = deg_df[mask].sort_values("imgsz", ascending=False)
        if sub.empty:
            ax.set_visible(False)
            continue

        band = float(sub["idsw_noise_band"].iloc[0])

        ax.plot(sub["imgsz"], sub["idsw_delta_norm"], "s-", color=COLOR_IDSW,
                label="IDSW/GT-track", markersize=5, linewidth=1.6)
        ax.fill_between(sub["imgsz"], -band, band,
                        color=COLOR_IDSW, alpha=0.12, label="±1-switch floor")
        ax.plot(sub["imgsz"], sub["mt_delta_norm"], "^:", color=COLOR_MT,
                label="MT loss", markersize=5, linewidth=1.6)

        ax.axhline(0, color="grey", linewidth=0.7, alpha=0.5)
        ax.set_xlim(650, 310)
        ax.set_xticks(RESOLUTIONS)
        ax.tick_params(axis="x", rotation=45, labelsize=9)
        ax.tick_params(axis="y", labelsize=9)
        ax.grid(axis="y", linewidth=0.4, alpha=0.4)
        ax.set_ylim(-0.8, 1.4)

        # ── Detection-suppression annotation ──────────────────────────────────
        # Only annotate Moderate and Dense rows: in those scenes, detection
        # collapse at low resolution is a genuine effect (high pedestrian density
        # → recall drops → association pool shrinks → IDSW goes negative despite
        # worsening tracking). In the Sparse row, small negative IDSW excursions
        # are noise-level fluctuations, not collapse, and should not be labelled.
        if seq_name in SUPPRESS_ANNOT_SEQS:
            idsw_min_row = sub.loc[sub["idsw_delta_norm"].idxmin()]
            if float(idsw_min_row["idsw_delta_norm"]) < SUPPRESS_ANNOT_THRESH:
                ax.annotate(
                    "↓ recall\nsuppresses\nIDSW",
                    xy=(float(idsw_min_row["imgsz"]), float(idsw_min_row["idsw_delta_norm"])),
                    xytext=(float(idsw_min_row["imgsz"]) + 60, float(idsw_min_row["idsw_delta_norm"]) - 0.18),
                    fontsize=7,
                    color=COLOR_IDSW,
                    ha="center",
                    arrowprops=dict(arrowstyle="->", color=COLOR_IDSW, lw=0.8),
                )

        # Column headers: model names — top row only
        if row_idx == 0:
            ax.set_title(MODEL_LABELS[model_path], fontsize=11, fontweight="bold", pad=6)

        # Row labels: sequence descriptors on the left y-axis only
        if col_idx == 0:
            ax.set_ylabel(
                f"{SEQ_LABELS[seq_name]}\n\n"
                "Relative change from 640 baseline\n"
                "(0 = baseline, positive = worse)",
                fontsize=9,
            )

        # x-axis label: bottom row only
        if row_idx == n_rows - 1:
            ax.set_xlabel("Input resolution (px)", fontsize=10)

        # Legend: top-left panel only, larger font for IEEE reproduction
        if row_idx == 0 and col_idx == 0:
            ax.legend(fontsize=9, loc="upper left", framealpha=0.95)

out_path = RESULTS_RAW.parent / "figures" / "exp2_degradation_curves.pdf"
plt.savefig(out_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved → {out_path}")

"MT loss is computed as (MT₆₄₀ − MT_res) / MT₆₄₀, where MT₆₄₀ is the 640px baseline, which is itself below 1.0 for all conditions. MOT17-09 nano achieves MT₆₄₀ = 0.62, reflecting the difficulty of ByteTrack association under low-angle geometry even at full resolution."

### Next Steps

1. Incorporate MOT20 elevated-angle sequences to decouple camera geometry from resolution effects
2. Explore per-sequence ByteTrack tuning and report sensitivity of the operating envelope to `match_thresh`
3. Repeat on edge devices (RPi 5, Jetson Nano, Jetson Orin Nano) to characterize device-specific envelopes

In [ ]:
# Sanity-check video — MOT17-04 at 640px baseline (yolo26s)
#
# Replays bounding boxes and track IDs from the existing CSV onto the original
# sequence frames and writes a full-resolution mp4.  No model re-inference.
# yolo26s is used as a representative mid-size variant for visual inspection.
# OpenCV writes mp4v (MPEG-4 Part 2); ffmpeg re-encodes to H.264 for QuickTime
# and browser compatibility.

from IPython.display import Video, display
from benchmark.preview import render_tracking_video, transcode_h264
from benchmark.mot_gt import load_seqinfo

PREVIEW_MODEL = PROFILE.model_variants[1]   # yolo26s — mid-size representative
SEQ_NAME      = "MOT17-04"
PREVIEW_CSV   = RESULTS_RAW / PROFILE.result_tag / "exp1" / f"{SEQ_NAME}_{PREVIEW_MODEL}_{IMGSZ_BASE}_{PROFILE.result_tag}.csv"
SEQ_DIR       = DATA_ROOT / f"{SEQ_NAME}-{SEQ_SUFFIX}"
PREVIEW_DIR   = RESULTS_RAW.parent / "preview"
OUT_RAW       = PREVIEW_DIR / f"{SEQ_NAME}_{PREVIEW_MODEL}_{IMGSZ_BASE}_raw.mp4"
OUT_H264      = PREVIEW_DIR / f"{SEQ_NAME}_{PREVIEW_MODEL}_{IMGSZ_BASE}.mp4"

fps = load_seqinfo(SEQ_DIR)["frameRate"]

print(f"Rendering {PREVIEW_CSV.name} → {OUT_RAW.name} ...")
render_tracking_video(PREVIEW_CSV, SEQ_DIR / "img1", OUT_RAW, fps=fps)
print(f"  raw mp4v: {OUT_RAW.stat().st_size / 1e6:.1f} MB")

print(f"Transcoding to H.264 → {OUT_H264.name} ...")
transcode_h264(OUT_RAW, OUT_H264)
print(f"  H.264:    {OUT_H264.stat().st_size / 1e6:.1f} MB")

display(Video(str(OUT_H264), embed=False, width=960))